# 01 · Data Collection

This step produces the raw inputs for every later step:

1. a **validated product catalog** (`data/raw/products.jsonl`)
2. **intent labels** for the classifier (`data/labeled/intent_queries.csv`)
3. **relevance judgments** for IR evaluation (`data/labeled/relevance_judgments.csv`)
4. the catalog **stored in PostgreSQL** (Neon)

The products were already scraped in the `product_search_and_analysis` project (8 sources, merged into one
JSONL file). No scraping happens here.

Cells that write files or the database are guarded by flags (`APPEND_LABELS`, `WRITE_JUDGMENTS`, `LOAD_TO_DB`)
so the notebook is safe to re-run.

In [ ]:
import csv
import json
import os
import re
from pathlib import Path

import pandas as pd

# Paths. Move these to src/catalogsearch/config.py once it exists.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_PRODUCTS = ROOT / "data" / "raw" / "products.jsonl"
LABELED_DIR = ROOT / "data" / "labeled"
INTENT_QUERIES = LABELED_DIR / "intent_queries.csv"
RELEVANCE_JUDGMENTS = LABELED_DIR / "relevance_judgments.csv"
JUDGING_POOL = LABELED_DIR / "judging_pool.csv"
SCHEMA_FILE = ROOT / "db" / "schema.sql"

INTENT_LABELS = ("search", "info")
RELEVANCE_LEVELS = (0, 1, 2, 3)  # 0 irrelevant, 1 marginal, 2 relevant, 3 highly relevant

pd.set_option("display.max_colwidth", 70)
print("project root:", ROOT)

## 1. Load the scraped catalog (data/raw/products.jsonl)

In [ ]:
REQUIRED = ["product_id", "title", "description", "category", "source"]  # NOT NULL in the products table
OPTIONAL = ["brand", "price", "specs", "features", "image_url", "source_url", "rating"]


def load_jsonl(path):
    """Return (records, bad_lines). A malformed line is reported, not fatal."""
    records, bad = [], []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                bad.append((line_no, exc.msg))
    return records, bad


products, bad_lines = load_jsonl(RAW_PRODUCTS)
df = pd.DataFrame(products)
print(f"{len(products)} products loaded, {len(bad_lines)} unparseable lines")
df.head(3)

### Validate against the `products` schema

In [ ]:
def is_empty(value):
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    if isinstance(value, (str, list, dict)):
        return len(value) == 0 if not isinstance(value, str) else value.strip() == ""
    return False


missing_columns = sorted(set(REQUIRED + OPTIONAL) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(REQUIRED + OPTIONAL))
empty_required = {c: int(df[c].map(is_empty).sum()) for c in REQUIRED if c in df}
duplicate_ids = int(df["product_id"].duplicated().sum())
bad_types = {
    "price": int((~df["price"].map(lambda v: v is None or isinstance(v, (int, float)))).sum()),
    "rating": int((~df["rating"].map(lambda v: v is None or isinstance(v, (int, float)))).sum()),
    "specs": int((~df["specs"].map(lambda v: isinstance(v, dict))).sum()),
    "features": int((~df["features"].map(lambda v: isinstance(v, list))).sum()),
}

print("missing columns :", missing_columns or "none")
print("extra columns   :", extra_columns or "none")
print("empty required  :", empty_required)
print("duplicate ids   :", duplicate_ids)
print("wrong types     :", bad_types)

assert not missing_columns, "catalog is missing schema columns"
assert duplicate_ids == 0, "product_id must be unique"
assert not any(empty_required.values()), "required fields must not be empty"
assert not any(bad_types.values()), "unexpected field types"
print("\ncatalog is valid for the products table")

### What was collected

In [ ]:
by_source = df["source"].value_counts().rename("products").to_frame()
by_source["share_%"] = (by_source["products"] / len(df) * 100).round(1)
by_source

In [ ]:
df["category"].value_counts().rename("products").to_frame()

In [ ]:
coverage = pd.DataFrame({c: df[c].map(is_empty) for c in REQUIRED + OPTIONAL}).mean()
(coverage * 100).round(1).sort_values(ascending=False).rename("% empty").to_frame()

> Hand-off to **02 Data analysis**: the catalog has more categories than the six the project targets
> (phones, laptops, PCs, headphones, chargers, power banks), and `rating` / `brand` are often empty.

## 2. Collect intent labels (search / info)

**Labelling rule:** would a ranked list of products fully satisfy the user?
Yes → `search` (product names, models, specs, filters). No, they need advice or an explanation → `info`.

| query | label |
|---|---|
| `sony wh-1000xm5`, `65 inch 4k smart tv`, `cheap usb-c cable` | search |
| `best budget phone for students`, `is 120hz worth it`, `oled or qled` | info |

In [ ]:
def load_intent_labels(path=INTENT_QUERIES):
    labels = pd.read_csv(path, dtype=str, keep_default_na=False)
    labels["query"] = labels["query"].str.strip()
    labels["label"] = labels["label"].str.strip().str.lower()
    invalid = labels[~labels["label"].isin(INTENT_LABELS)]
    if len(invalid):
        raise ValueError(f"invalid labels in {path.name}:\n{invalid}")
    labels = labels[labels["query"] != ""]
    return labels.loc[~labels["query"].str.lower().duplicated()].reset_index(drop=True)


def append_intent_labels(rows, path=INTENT_QUERIES):
    """Append (query, label) pairs. Skips queries already in the file (case-insensitive)."""
    existing = set(load_intent_labels(path)["query"].str.lower())
    to_write, seen = [], set()
    for query, label in rows:
        query, label = query.strip(), label.strip().lower()
        if label not in INTENT_LABELS:
            raise ValueError(f"invalid label {label!r} for {query!r}")
        key = query.lower()
        if query and key not in existing and key not in seen:
            seen.add(key)
            to_write.append((query, label))
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(to_write)
    return len(to_write)


labels = load_intent_labels()
print(f"{len(labels)} labelled queries")
labels["label"].value_counts().rename("queries").to_frame()

### 2a. Hand-written queries
Add queries that real users of this catalog would type. Keep the two labels roughly balanced.

In [ ]:
NEW_LABELS = [
    # search: the user names a product, model, spec or category
    ("samsung galaxy a55 5g", "search"),
    ("xiaomi redmi note 13 pro", "search"),
    ("ipad air m2 11 inch", "search"),
    ("jbl tune 520bt", "search"),
    ("anker 10000mah power bank", "search"),
    ("lenovo ideapad slim 3", "search"),
    ("apple watch series 9 45mm", "search"),
    ("logitech mx master 3s", "search"),
    ("65 inch 4k smart tv", "search"),
    ("usb-c to lightning cable", "search"),
    # info: the user needs advice, a comparison or an explanation
    ("what is the difference between an ipad and an android tablet?", "info"),
    ("how much ram do I need for video editing", "info"),
    ("is a 120hz screen worth it on a phone?", "info"),
    ("which smartwatch works best with an iphone", "info"),
    ("can I use a laptop charger to charge my phone?", "info"),
    ("why does my power bank charge so slowly?", "info"),
    ("best budget phone for students", "info"),
    ("should I buy an oled or qled tv", "info"),
    ("do I need a case for my phone", "info"),
    ("how to choose a gaming monitor", "info"),
]
pd.DataFrame(NEW_LABELS, columns=["query", "label"])["label"].value_counts()

### 2b. Search queries drawn from the catalog
Short versions of real product titles are realistic `search` queries. **Review them before appending**: drop
anything that would not be a plausible user query. They only add `search` examples, so add `info`
examples in 2a to keep the classes balanced.

In [ ]:
PER_CATEGORY = 1   # candidates per category
MAX_WORDS = 5      # users type short queries


def title_to_query(title, max_words=MAX_WORDS):
    words = [w.strip("-.+") for w in re.sub(r"[^\w\s+.-]", " ", title.lower()).split()]
    return " ".join([w for w in words if w][:max_words])


catalog_candidates = (
    df.sample(frac=1, random_state=42)
      .groupby("category")
      .head(PER_CATEGORY)
      .assign(query=lambda d: d["title"].map(title_to_query), label="search")
      [["query", "label", "category", "title"]]
      .drop_duplicates("query")
      .sort_values("category")
      .reset_index(drop=True)
)
catalog_candidates

In [ ]:
# Drop rows that are not plausible user queries, e.g. catalog_candidates.drop(index=[3, 7])
REJECT_ROWS = []
reviewed_candidates = catalog_candidates.drop(index=REJECT_ROWS)
print(f"{len(reviewed_candidates)} catalog queries kept")

### 2c. Append to `intent_queries.csv`

In [ ]:
APPEND_LABELS = False   # set True after reviewing 2a and 2b

rows = NEW_LABELS + list(reviewed_candidates[["query", "label"]].itertuples(index=False, name=None))
if APPEND_LABELS:
    written = append_intent_labels(rows)
    print(f"appended {written} new rows (duplicates skipped)")
else:
    print(f"{len(rows)} rows ready; set APPEND_LABELS = True to write them")

labels = load_intent_labels()
print(f"\n{len(labels)} labelled queries in total")
labels["label"].value_counts().rename("queries").to_frame()

## 3. Collect relevance judgments

The seed `relevance_judgments.csv` uses illustrative product IDs (`P102`, …). The check below shows how many
of its IDs exist in the real catalog. Replace them with real judgments:

1. **Pool**: for each judgment query, collect candidate products by keyword overlap with the title, brand and category.
2. **Judge**: open `data/labeled/judging_pool.csv` (Excel or Sheets) and fill the `relevance` column with 0–3.
3. **Save**: the judged rows become the new `relevance_judgments.csv`.

In [ ]:
catalog_ids = set(df["product_id"])
current = pd.read_csv(RELEVANCE_JUDGMENTS, dtype={"product_id": str})
known = current["product_id"].isin(catalog_ids)
print(f"{len(current)} existing judgments, {known.sum()} reference products in the catalog")

In [ ]:
JUDGMENT_QUERIES = {
    1: "wireless headphones",
    2: "iphone 15 pro max",
    3: "65w usb-c charger",
    4: "gaming laptop rtx 4060",
    5: "20000mah power bank",
    6: "samsung galaxy tablet",
    7: "smartwatch with heart rate monitor",
    8: "mechanical keyboard",
}
POOL_SIZE = 15  # candidates judged per query


def tokens(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


doc_tokens = (df["title"] + " " + df["brand"] + " " + df["category"]).map(tokens)


def candidate_pool(query, k=POOL_SIZE):
    """Products sharing the most query words with title/brand/category. Only for pooling, not the real ranker."""
    q = tokens(query)
    overlap = doc_tokens.map(lambda t: len(q & t) / len(q) if q else 0.0)
    pool = df.assign(overlap=overlap.round(2))
    pool = pool[pool["overlap"] > 0].sort_values(["overlap", "rating"], ascending=[False, False]).head(k)
    return pool[["product_id", "title", "brand", "category", "price", "overlap"]]


candidate_pool(JUDGMENT_QUERIES[1])

In [ ]:
pool = pd.concat(
    [candidate_pool(q).assign(query_id=qid, query=q) for qid, q in JUDGMENT_QUERIES.items()],
    ignore_index=True,
)
pool["relevance"] = ""
pool = pool[["query_id", "query", "product_id", "title", "brand", "category", "price", "relevance"]]
print(pool.groupby("query").size().rename("candidates").to_string())

if JUDGING_POOL.exists():
    print(f"\n{JUDGING_POOL.name} already exists, so it is not overwritten (it may hold judged work).")
else:
    pool.to_csv(JUDGING_POOL, index=False)
    print(f"\nwrote {JUDGING_POOL.name}; fill the relevance column (0-3) and re-run the next cell")

In [ ]:
WRITE_JUDGMENTS = False   # set True once judging_pool.csv has relevance values

judged = pd.read_csv(JUDGING_POOL, dtype={"product_id": str})
judged = judged[judged["relevance"].notna() & (judged["relevance"].astype(str).str.strip() != "")]
judged["relevance"] = judged["relevance"].astype(float).astype(int)

problems = judged[~judged["relevance"].isin(RELEVANCE_LEVELS) | ~judged["product_id"].isin(catalog_ids)]
assert problems.empty, f"fix these rows in {JUDGING_POOL.name}:\n{problems}"

print(f"{len(judged)} judged rows across {judged['query_id'].nunique()} queries")
if WRITE_JUDGMENTS and len(judged):
    judged[["query_id", "query", "product_id", "relevance"]].to_csv(RELEVANCE_JUDGMENTS, index=False)
    print(f"replaced {RELEVANCE_JUDGMENTS.name} with real judgments")
elif len(judged):
    print("set WRITE_JUDGMENTS = True to save them")
judged.pivot_table(index="query", columns="relevance", values="product_id", aggfunc="count", fill_value=0) if len(judged) else None

## 4. Store products in PostgreSQL (Neon)

Needs `DATABASE_URL` in `.env` (copy `.env.example`). The upsert is idempotent: re-running updates
existing rows instead of duplicating them. If `db/schema.sql` is empty, the schema below is used. It matches
`context.md` and the table created by `product_search_and_analysis`.

In [ ]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
DATABASE_URL = os.getenv("DATABASE_URL")
LOAD_TO_DB = True   # set False to skip the database entirely

DEFAULT_SCHEMA = """
CREATE TABLE IF NOT EXISTS products (
  product_id   TEXT PRIMARY KEY,
  title        TEXT NOT NULL,
  description  TEXT NOT NULL,
  brand        TEXT,
  category     TEXT NOT NULL,
  price        NUMERIC,
  specs        JSONB DEFAULT '{}'::jsonb,
  features     TEXT[] DEFAULT '{}',
  image_url    TEXT,
  source_url   TEXT,
  rating       NUMERIC,
  source       TEXT NOT NULL,
  collected_at TIMESTAMPTZ DEFAULT now()
);
CREATE INDEX IF NOT EXISTS idx_products_category ON products(category);
"""
schema_text = SCHEMA_FILE.read_text(encoding="utf-8").strip() if SCHEMA_FILE.exists() else ""
schema_sql = schema_text or DEFAULT_SCHEMA
print("schema from:", "db/schema.sql" if schema_text else "DEFAULT_SCHEMA in this notebook")
print("database   :", "configured" if DATABASE_URL else "DATABASE_URL not set, so section 4 will be skipped")

In [ ]:
COLUMNS = ["product_id", "title", "description", "brand", "category", "price",
           "specs", "features", "image_url", "source_url", "rating", "source"]


def to_row(p):
    """One product as a tuple in COLUMNS order. Empty strings become NULL for optional text fields."""
    from psycopg2.extras import Json

    def text_or_none(v):
        return v.strip() if isinstance(v, str) and v.strip() else None

    return (
        p["product_id"], p["title"], p["description"], text_or_none(p.get("brand")), p["category"],
        p.get("price"), Json(p.get("specs") or {}), [str(f) for f in (p.get("features") or [])],
        text_or_none(p.get("image_url")), text_or_none(p.get("source_url")), p.get("rating"), p["source"],
    )


UPSERT_SQL = (
    f"INSERT INTO products ({', '.join(COLUMNS)}) VALUES %s "
    "ON CONFLICT (product_id) DO UPDATE SET "
    + ", ".join(f"{c} = EXCLUDED.{c}" for c in COLUMNS if c != "product_id")
)

if LOAD_TO_DB and DATABASE_URL:
    import psycopg2
    from psycopg2.extras import execute_values

    conn = psycopg2.connect(DATABASE_URL)
    try:
        with conn, conn.cursor() as cur:          # one transaction: all rows or none
            cur.execute(schema_sql)
            execute_values(cur, UPSERT_SQL, [to_row(p) for p in products], page_size=500)
        print(f"upserted {len(products)} products")
    finally:
        conn.close()
else:
    print("skipped: set DATABASE_URL in .env and LOAD_TO_DB = True")

### Verify the database matches the file

In [ ]:
if LOAD_TO_DB and DATABASE_URL:
    conn = psycopg2.connect(DATABASE_URL)
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT source, COUNT(*) FROM products GROUP BY source")
            db_counts = dict(cur.fetchall())
    finally:
        conn.close()
    check = df["source"].value_counts().rename("file").to_frame()
    check["database"] = check.index.map(lambda s: db_counts.get(s, 0))
    check["ok"] = check["database"] >= check["file"]
    extra_sources = sorted(set(db_counts) - set(check.index))
    if extra_sources:
        print("sources only in the database:", extra_sources)
    display(check)
else:
    print("skipped")

## Summary

In [ ]:
labels = load_intent_labels()
judged_rows = len(pd.read_csv(RELEVANCE_JUDGMENTS))
print(f"products        : {len(products)} from {df['source'].nunique()} sources, {df['category'].nunique()} categories")
print(f"intent labels   : {len(labels)} ({labels['label'].value_counts().to_dict()})")
print(f"judgments       : {judged_rows} rows in {RELEVANCE_JUDGMENTS.name}")
print(f"postgres        : {'loaded' if LOAD_TO_DB and DATABASE_URL else 'not loaded'}")
print("\nnext: notebooks/02_data_analysis.ipynb")